# Phase 0 — Data pipeline

**Project:** Latent mood & user-taste modelling with MSD audio features and Last.fm listening histories.

**Goal of this notebook:** turn the two raw datasets into the two clean CSVs that every later phase consumes:

- `data/songs_clean.csv` — one row per matched song, six audio features normalised to $[0,1]$.
- `data/listens_clean.csv` — one row per (user, song) pair, with `play_count` and `listened` (0/1).

**Inputs** (must already be extracted on disk):

- `data/lastfm1k/userid-timestamp-artid-artname-traid-traname.tsv` (~1.3 GB)
- `data/msd/MillionSongSubset/A/B/C/TR…h5` (10 000 files, ~2.6 GB)

**Pipeline** (joins on normalised `(artist_name, track_title)`):

1. Load Last.fm scrobbles → DataFrame.
2. Walk MSD HDF5 tree, pull six audio features per song → DataFrame.
3. Normalise artist/track strings on both sides.
4. Inner-join MSD × Last.fm — matched corpus.
5. Aggregate listen events; keep users with ≥5 listens in the matched corpus.
6. Sample 5 negatives per positive listen.
7. MinMax-scale the six features to $[0,1]$.
8. Save the two CSVs.
9. EDA: corpus size, listen-count distribution, feature correlation heatmap, feature histograms.


## 0  Imports & global config


In [ ]:
import os
import h5py
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import MinMaxScaler
from pathlib import Path

# Random seed and plotting setup
np.random.seed(42)
plt.style.use('ggplot')
%matplotlib inline
plt.rcParams['figure.figsize'] = (12, 8)

# Pointing to your specific local data directory
DATA_DIR = Path('/Users/jacobbrams/Desktop/MBML_data')
LASTFM_DIR = DATA_DIR / 'lastfm-dataset-1K'
MSD_DIR = DATA_DIR / 'MillionSongSubset'

# Features to extract
FEATURE_COLS = ['loudness', 'tempo', 'energy', 'danceability', 'mode', 'time_signature']

## 1  Load Last.fm 1K listening histories

The TSV is ~1.3 GB and ~19 M rows — every scrobble of every user. We load just the columns we need and parse timestamps. A few rows are malformed (uneven number of fields); we skip them silently.


In [ ]:
lastfm_path = LASTFM_DIR / 'userid-timestamp-artid-artname-traid-traname.tsv'
df_lastfm = pd.read_csv(
    lastfm_path, sep='\t', header=None,
    names=['user_id', 'timestamp', 'artist_id', 'artist_name', 'track_id', 'track_name'],
    parse_dates=['timestamp'],
    on_bad_lines='skip',
)
print(f'Last.fm scrobbles: {len(df_lastfm):,} rows')
print(f'Unique users:      {df_lastfm["user_id"].nunique():,}')
df_lastfm.head()


## 2  Load the MSD 10K subset

Each `.h5` file holds one song. We walk the `A/B/C/TR…h5` directory tree, open every file, and pull six pre-computed Echo Nest features (the same features Spotify used internally before they deprecated the public API). This is the slow step: ~10 000 small files, expect 1–3 minutes on an M-series Mac.


In [ ]:
def load_msd_track(filepath):
    """Return one song's metadata + audio features as a flat dict."""
    with h5py.File(filepath, 'r') as f:
        meta = f['metadata']['songs']
        ana = f['analysis']['songs']
        return {
            'song_id':        meta['song_id'][0].decode(),
            'artist_name':    meta['artist_name'][0].decode(),
            'title':          meta['title'][0].decode(),
            'loudness':       float(ana['loudness'][0]),
            'tempo':          float(ana['tempo'][0]),
            'time_signature': int(ana['time_signature'][0]),
            'mode':           int(ana['mode'][0]),
            'energy':         float(ana['energy'][0]),
            'danceability':   float(ana['danceability'][0]),
        }

records = []
for dirpath, _, files in os.walk(MSD_DIR):
    for fname in files:
        if fname.endswith('.h5'):
            try:
                records.append(load_msd_track(os.path.join(dirpath, fname)))
            except Exception as e:
                pass  # skip the rare unreadable file
df_msd = pd.DataFrame(records)
print(f'MSD songs loaded: {len(df_msd):,}')
df_msd.head()


## 3  Normalise artist & track strings

Last.fm and MSD both store artist + title as free-form strings. We lowercase, strip whitespace, and replace `-`/`_` with spaces so spelling differences don't break the join. (We do NOT strip punctuation — track titles legitimately contain commas, parentheses, etc.)


In [ ]:
def normalise_str(s):
    if not isinstance(s, str):
        return ''
    return s.lower().strip().replace('-', ' ').replace('_', ' ')

df_lastfm['artist_norm'] = df_lastfm['artist_name'].map(normalise_str)
df_lastfm['track_norm']  = df_lastfm['track_name'].map(normalise_str)
df_msd['artist_norm']    = df_msd['artist_name'].map(normalise_str)
df_msd['track_norm']     = df_msd['title'].map(normalise_str)


## 4  Inner-join MSD × Last.fm — matched corpus

Inner join on `(artist_norm, track_norm)`. We keep only songs that appear in BOTH datasets — this is the working corpus for every downstream phase.


In [ ]:
lastfm_pairs = df_lastfm[['artist_norm', 'track_norm']].drop_duplicates()
df_songs = df_msd.merge(lastfm_pairs, on=['artist_norm', 'track_norm'], how='inner')
df_songs = df_songs.drop_duplicates(subset='song_id').reset_index(drop=True)
print(f'Matched corpus size: {len(df_songs):,} songs')


## 5  Active users & positive listen events

Aggregate Last.fm scrobbles into `(user, song)` play counts within the matched corpus. Drop users with fewer than 5 listens — they don't carry enough signal for a per-user taste profile.


In [ ]:
df_events = df_lastfm[['user_id', 'artist_norm', 'track_norm']].merge(
    df_songs[['song_id', 'artist_norm', 'track_norm']],
    on=['artist_norm', 'track_norm'], how='inner'
)
df_listens_pos = (
    df_events.groupby(['user_id', 'song_id'])
             .size().reset_index(name='play_count')
)
df_listens_pos['listened'] = 1

user_counts = df_listens_pos.groupby('user_id')['song_id'].count()
active_users = user_counts[user_counts >= 5].index
df_listens_pos = df_listens_pos[df_listens_pos['user_id'].isin(active_users)]
print(f'Active users (>=5 listens in matched corpus): {len(active_users):,}')
print(f'Positive listen events:                       {len(df_listens_pos):,}')


## 6  Negative sampling at 5:1

For every positive listen $l_{us}=1$, we draw 5 songs the user did NOT listen to and label them $l_{us}=0$. This balances the implicit-feedback Bernoulli likelihood in later phases.


In [ ]:
all_song_ids = df_songs['song_id'].to_numpy()
negatives = []
for user_id in active_users:
    pos = set(df_listens_pos.loc[df_listens_pos['user_id'] == user_id, 'song_id'])
    neg_pool = [s for s in all_song_ids if s not in pos]
    n_neg = min(5 * len(pos), len(neg_pool))
    if n_neg == 0:
        continue
    sampled = np.random.choice(neg_pool, n_neg, replace=False)
    for s in sampled:
        negatives.append({'user_id': user_id, 'song_id': s, 'play_count': 0, 'listened': 0})
df_negs = pd.DataFrame(negatives)
df_listens = pd.concat([df_listens_pos, df_negs], ignore_index=True)
print(f'Total listen events (pos + neg): {len(df_listens):,}')
print(f'  positives: {(df_listens["listened"]==1).sum():,}')
print(f'  negatives: {(df_listens["listened"]==0).sum():,}')


## 7  Feature normalisation

Drop rows missing any of the six features, then MinMax-scale them to $[0,1]$. Same scaling for every feature so the multivariate Gaussian likelihood in Phase 1+ is well-conditioned.


In [ ]:
df_songs = df_songs.dropna(subset=FEATURE_COLS).reset_index(drop=True)
scaler = MinMaxScaler()
df_songs[FEATURE_COLS] = scaler.fit_transform(df_songs[FEATURE_COLS])
df_songs[FEATURE_COLS].describe().T


## 8  Save the two clean CSVs

These two files are the inputs for every later phase. They are committed to the repo so the model notebook can be re-run without redoing Phase 0.


In [ ]:
songs_out = DATA_DIR / 'songs_clean.csv'
listens_out = DATA_DIR / 'listens_clean.csv'

songs_keep = ['song_id', 'artist_name', 'title'] + FEATURE_COLS
df_songs[songs_keep].to_csv(songs_out, index=False)
df_listens[['user_id', 'song_id', 'play_count', 'listened']].to_csv(listens_out, index=False)

print(f'Wrote {songs_out}: {len(df_songs):,} songs')
print(f'Wrote {listens_out}: {len(df_listens):,} listen events')


## 9  Exploratory data analysis

### 9.1 Listen-count distribution per user

Heavy right tail is expected — a handful of power users with thousands of listens, a long tail of light users.


In [ ]:
user_listens = df_listens_pos.groupby('user_id').size()
sns.histplot(user_listens, bins=40, log_scale=(False, True))
plt.xlabel('Number of positive listen events per user')
plt.ylabel('User count (log)')
plt.title(f'Positive listens per user ({len(active_users):,} active users)')
plt.show()

print(user_listens.describe())


### 9.2 Feature correlation heatmap

Check that the six audio features are not perfectly collinear (which would cause issues with the multivariate-Gaussian covariance in Phase 1).


In [ ]:
corr = df_songs[FEATURE_COLS].corr()
sns.heatmap(corr, annot=True, fmt='.2f', cmap='RdBu_r', center=0, vmin=-1, vmax=1)
plt.title('Audio feature correlations (after MinMax-scaling)')
plt.show()


### 9.3 Feature histograms

Per-feature distribution after scaling — useful sanity check that `mode` and `time_signature` (categorical-ish features) didn't get crushed by the scaler.


In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(14, 8))
for ax, col in zip(axes.flat, FEATURE_COLS):
    sns.histplot(df_songs[col], bins=30, ax=ax)
    ax.set_title(col)
    ax.set_xlabel('')
plt.tight_layout()
plt.show()


### 9.4 Summary

Final printout for the Phase 0 gate (master prompt requires this before moving to Phase 1):


In [ ]:
print('=' * 60)
print('PHASE 0 SUMMARY')
print('=' * 60)
print(f'Matched corpus size:        {len(df_songs):,} songs')
print(f'Active users (>=5 listens): {len(active_users):,}')
print(f'Positive listen events:     {(df_listens["listened"]==1).sum():,}')
print(f'Negative listen events:     {(df_listens["listened"]==0).sum():,}')
print()
print('Per-feature stats (MinMax-scaled):')
print(df_songs[FEATURE_COLS].describe().T[['mean', 'std', 'min', 'max']])
